# Tunix-Med: Final Knowledge Evaluation

Evaluates the fine-tuned `google/gemma-3-1b-it` model on questions sampled
**directly from the training dataset** (`lmassaron/medical-cardiology-qa`).

The adapter loaded from `tunix-medical-model/` is the **best checkpoint**
from training — `CleanProgressHook` in notebook 03 overwrites that directory
only when eval loss improves, so the files on disk always reflect the best
step rather than the final step.

1. Reconstructs the held-out split (seed=42, 10% eval) from training.
2. Samples 300 diverse questions from the held-out set.
3. Uses the **same system prompt** the model was trained with.
4. Scores with keyword match, semantic similarity, and an AI judge.

In [1]:
import os, warnings, re
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings("ignore")


def info_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = info_device()
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float32
)

BASE_MODEL = "google/gemma-3-1b-it"
# CleanProgressHook in 03 overwrites this directory every time eval loss
# improves, so this always contains the BEST checkpoint — not the final step.
ADAPTER_PATH = "tunix-medical-model"

print(f"Device : {device}  |  dtype : {dtype}")
print(f"Loading best adapter from : {ADAPTER_PATH}/")

import os, json as _json

cfg_path = os.path.join(ADAPTER_PATH, "adapter_config.json")
if os.path.exists(cfg_path):
    cfg = _json.load(open(cfg_path))
    print(f"  base model     : {cfg.get('base_model_name_or_path')}")
    print(f"  LoRA rank      : {cfg.get('r')}  alpha={cfg.get('lora_alpha')}")
    print(f"  target modules : {cfg.get('target_modules')}")
else:
    print("  ⚠  adapter_config.json not found — run Cell 8 of notebook 03 first.")

print("Loading base model ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    device_map=device,
)
print("Loading LoRA adapter ...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print("Merging adapter into base weights ...")
model = model.merge_and_unload()
model.eval()
print("Model ready.")

Device : cuda  |  dtype : torch.bfloat16
Loading best adapter from : tunix-medical-model/
  base model     : google/gemma-3-1b-it
  LoRA rank      : 16  alpha=32
  target modules : ['gate_proj', 'up_proj', 'down_proj', 'q_proj', 'kv_proj', 'o_proj']
Loading base model ...
Loading LoRA adapter ...
Merging adapter into base weights ...
Model ready.


## Sample Evaluation Questions from the Dataset

We reconstruct the held-out eval split (seed=42, 10% split) and draw 25
questions with broad topic coverage — the same distribution the model trained on.

In [2]:
# ── Load the dataset and reconstruct the eval split ──────────────────────────
# Uses the same seed and split fraction as the training notebook so the
# questions are guaranteed to be from the held-out set the model never trained on.
DATASET_ID = "lmassaron/medical-cardiology-qa"
EVAL_SPLIT = 0.1
SEED = 42
N_EVAL_QS = 300  # number of questions to evaluate

print(f"Loading {DATASET_ID} ...")
full_ds = load_dataset(DATASET_ID, split="train")
n = len(full_ds)

rng = np.random.default_rng(SEED)
all_idx = rng.permutation(n)
cut = int(n * (1.0 - EVAL_SPLIT))
eval_idx = all_idx[cut:]  # same held-out indices as training

print(f"  Total rows  : {n:,}")
print(f"  Eval rows   : {len(eval_idx):,}")


# ── Extract Q-A pairs from the messages format ────────────────────────────────
def extract_qa(example):
    msgs = example["messages"]
    # Format: [system, user, assistant]
    user_msg = next(m["content"] for m in msgs if m["role"] == "user")
    asst_msg = next(m["content"] for m in msgs if m["role"] == "assistant")
    return {"question": user_msg, "answer": asst_msg}


# ── Sample 25 questions with topic diversity ──────────────────────────────────
# Shuffle the eval indices and take the first N_EVAL_QS that have
# non-trivial answers (length > 20 chars) and varied question starts.
rng2 = np.random.default_rng(SEED + 1)
shuffled = rng2.permutation(eval_idx)

seen_prefixes = set()
qa_pairs = []
for idx in shuffled:
    if len(qa_pairs) >= N_EVAL_QS:
        break
    ex = extract_qa(full_ds[int(idx)])
    q, a = ex["question"], ex["answer"]
    # Skip very short answers (likely yes/no)
    if len(a) < 25:
        continue
    # Ensure diversity: track first 4 words of question as a loose topic key
    prefix = " ".join(q.lower().split()[:4])
    if prefix in seen_prefixes:
        continue
    seen_prefixes.add(prefix)
    qa_pairs.append({"question": q, "answer": a, "dataset_idx": int(idx)})

data = pd.DataFrame(qa_pairs)
print(f"\nSampled {len(data)} evaluation questions")
print("\nSample questions:")
for _, row in data.head(5).iterrows():
    print(f"  Q: {row['question'][:90]}")
    print(f"  A: {row['answer'][:80]}")
    print()

Loading lmassaron/medical-cardiology-qa ...
  Total rows  : 10,518
  Eval rows   : 1,052

Sampled 300 evaluation questions

Sample questions:
  Q: What is the rate of bleeding stroke associated with statin use over 5 years of treatment p
  A: Over 5 years of treatment, statins result in 7.5 cases of bleeding stroke per 10

  Q: Which medications used in cardiac stress testing can cause mild hypotension?
  A: Adenosine and dipyridamole.

  Q: What is the advantage of using perfusion stress test with 99mTc labelled sestamibi?
  A: It is appropriate for select patients with an abnormal resting electrocardiogram

  Q: What is the suggested course of action if angina is suspected?
  A: An urgent medical assessment is suggested to rule out serious medical conditions

  Q: What is the benefit of using beta blockers in glaucoma treatment?
  A: They can lower intraocular pressure.



## Inference Loop

In [3]:
# The system prompt used during training — must match exactly
SYSTEM_PROMPT = (
    "You are a knowledgeable medical assistant specializing in cardiology. "
    "Answer clinical questions accurately, focusing on diagnostic criteria, "
    "treatment guidelines, and pathophysiology."
)

results_list = []
model.eval()

for _, row in tqdm(data.iterrows(), total=len(data)):
    question = row["question"]
    answer = row["answer"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    input_ids = encoded.to(device)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    gen_ids = outputs[0, input_ids.shape[-1] :]
    generated = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # Reference perplexity (how surprised the model is by the correct answer)
    ref_ids = tokenizer(
        answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = torch.cat([input_ids, ref_ids], dim=1)
    labels = full_ids.clone()
    labels[:, : input_ids.shape[1]] = -100
    with torch.no_grad():
        loss = model(
            full_ids, attention_mask=torch.ones_like(full_ids), labels=labels
        ).loss
    perplexity = torch.exp(loss).item()

    results_list.append(
        {
            "question": question,
            "expected_answer": answer,
            "generated_answer": generated,
            "perplexity": perplexity,
        }
    )

results_df = pd.DataFrame(results_list)
print(f"Generated {len(results_df)} answers")
results_df[["question", "expected_answer", "generated_answer", "perplexity"]].head()

100%|██████████| 300/300 [01:26<00:00,  3.45it/s]

Generated 300 answers


,question,expected_answer,generated_answer,perplexity
0,What is the rate of bleeding stroke associated...,"Over 5 years of treatment, statins result in 7...","About 2 cases per 10,000 people treated.",2.228864
1,Which medications used in cardiac stress testi...,Adenosine and dipyridamole.,Vasodilators such as nitroglycerin.,3.629328
2,What is the advantage of using perfusion stres...,It is appropriate for select patients with an ...,It allows for measurement of myocardial perfus...,37.118767
3,What is the suggested course of action if angi...,An urgent medical assessment is suggested to r...,Immediate assessment for risk factors and card...,14.985295
4,What is the benefit of using beta blockers in ...,They can lower intraocular pressure.,They can help reduce intraocular pressure.,2.927745


## Scoring

In [4]:
# ════════════════════════════════════════════════════════════════════════════
# Metric 1 — Keyword F1 with TF-IDF weighting
# ════════════════════════════════════════════════════════════════════════════
# Problems with the old metric:
#   • Pure recall (found/total) rewards verbose outputs: a model that writes
#     200 words finds more keywords by sheer volume, even if mostly wrong.
#   • All keywords treated equally — "the" and "neprilysin" scored the same.
#
# Fix:
#   • F1 of keywords = harmonic mean of precision and recall.
#     Precision = fraction of keywords in the generated answer that also
#     appear in the reference.  Recall = fraction of reference keywords
#     found in the generated answer.  A verbose wrong answer has low
#     precision; a short correct answer has low recall but high precision;
#     F1 rewards the right balance.
#   • TF-IDF weights: rare domain terms (e.g. drug names, anatomical terms)
#     contribute more than common words.  We fit a TF-IDF vectorizer over
#     the reference answers in the eval set and use its idf_ array to weight
#     each keyword's contribution.
# ════════════════════════════════════════════════════════════════════════════
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Fit TF-IDF on all reference answers so rare medical terms get higher weight
_tfidf = TfidfVectorizer(analyzer="word", token_pattern=r"\b\w{4,}\b",
                         sublinear_tf=True)
_tfidf.fit(results_df["expected_answer"].tolist())
_vocab   = _tfidf.vocabulary_          # word → column index
_idf     = _tfidf.idf_                 # column index → idf weight


def keyword_f1_tfidf(generated: str, expected: str) -> float:
    """
    TF-IDF-weighted keyword F1 between generated and reference answer.
    Returns a value in [0, 1].  Penalises verbose wrong answers.
    """
    ref_kws = set(re.findall(r"\b\w{4,}\b", expected.lower()))
    gen_kws = set(re.findall(r"\b\w{4,}\b", generated.lower()))
    if not ref_kws:
        return 1.0

    def weighted_count(kws_to_check, universe):
        total = 0.0
        for w in universe:
            if w in kws_to_check:
                total += _idf[_vocab[w]] if w in _vocab else 1.0
        return total

    ref_weight = sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in ref_kws)
    gen_weight = sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in gen_kws) if gen_kws else 0.0

    if ref_weight == 0 or gen_weight == 0:
        return 0.0

    recall    = weighted_count(gen_kws, ref_kws) / ref_weight
    precision = weighted_count(ref_kws, gen_kws) / gen_weight
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ════════════════════════════════════════════════════════════════════════════
# Metric 2 — Calibrated semantic similarity
# ════════════════════════════════════════════════════════════════════════════
# Problem: cosine similarity from sentence-BERT clusters in a narrow band
# [~0.3, ~1.0] for any pair of topically related medical sentences.
# A wrong answer scores ~0.5-0.65, which looks like "partial credit" but is
# essentially chance-level for medical text.
#
# Fix: stretch the score to the full [0, 1] range using the empirical
# minimum and maximum across all (generated, expected) pairs in the eval set.
# score_calibrated = (raw - min_raw) / (max_raw - min_raw)
# We compute raw scores first, then calibrate in one pass.
# ════════════════════════════════════════════════════════════════════════════
from sentence_transformers import SentenceTransformer, util

sim_model = SentenceTransformer("all-MiniLM-L6-v2")


def _raw_semantic(generated: str, expected: str) -> float:
    e1 = sim_model.encode(generated, convert_to_tensor=True)
    e2 = sim_model.encode(expected,  convert_to_tensor=True)
    return float(util.pytorch_cos_sim(e1, e2))


# ════════════════════════════════════════════════════════════════════════════
# Metric 3 — Improved AI judge with chain-of-thought
# ════════════════════════════════════════════════════════════════════════════
# Problems with old prompt:
#   • No verbosity penalty — long wrong answers scored as "partial"
#   • No factual specificity requirement — "sounds medical" ≠ correct
#   • Returning ONLY a number forces commitment before reasoning
#
# Fix:
#   • Prompt the judge to reason first (brief CoT), then output the score
#     on a LAST LINE by itself — we parse only that last line.
#   • Explicit rubric: factual correctness > plausibility; verbosity that
#     buries the correct answer is penalised.
#   • max_new_tokens raised to 120 to allow brief reasoning.
# ════════════════════════════════════════════════════════════════════════════
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
)
judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_mdl = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, quantization_config=bnb_cfg, device_map="auto",
)


def ai_judge(question: str, generated: str, expected: str) -> float:
    """
    Returns a score in [0, 1].
    The judge reasons briefly then outputs an integer 1-10 on the last line.
    """
    prompt = (
        "You are a strict medical fact-checker.\n"
        "Evaluate ONLY factual correctness of the generated answer "
        "against the reference answer for the given question.\n\n"
        f"Question  : {question}\n"
        f"Reference : {expected}\n"
        f"Generated : {generated}\n\n"
        "Rules:\n"
        "- Factual accuracy is paramount. A plausible-sounding but wrong "
        "answer scores 1-3.\n"
        "- Specific facts matter: correct drug names, numbers, and mechanisms "
        "are required for a high score.\n"
        "- A correct answer that is unnecessarily verbose scores at most 7.\n"
        "- A concise, fully correct answer scores 9-10.\n"
        "- A partially correct answer (right concept, missing key detail) "
        "scores 4-6.\n\n"
        "First write ONE sentence explaining your reasoning.\n"
        "Then on the very last line write ONLY the integer score (1-10)."
    )
    inp  = judge_tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(judge_mdl.device)
    mask = torch.ones_like(inp)
    with torch.no_grad():
        out = judge_mdl.generate(
            inp, attention_mask=mask,
            max_new_tokens=120,   # enough for one reasoning sentence + score
            do_sample=False,
        )
        txt = judge_tok.decode(out[0, inp.shape[-1]:],
                               skip_special_tokens=True).strip()
    # Parse: take the last non-empty line and extract the first integer
    last_line = next((l.strip() for l in reversed(txt.splitlines()) if l.strip()), "")
    m = re.search(r"\b(\d+)\b", last_line)
    if m:
        score = int(m.group(1))
        return min(max(score, 1), 10) / 10.0
    # Fallback: search anywhere in the output
    m2 = re.search(r"\b(\d+)\b", txt)
    return min(max(int(m2.group(1)), 1), 10) / 10.0 if m2 else 0.5


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
# ── Compute raw scores ────────────────────────────────────────────────────────
print("Computing keyword F1 (TF-IDF weighted) ...")
results_df["keyword_score"] = results_df.apply(
    lambda r: keyword_f1_tfidf(r["generated_answer"], r["expected_answer"]), axis=1)

print("Computing raw semantic similarity ...")
results_df["_raw_sim"] = results_df.apply(
    lambda r: _raw_semantic(r["generated_answer"], r["expected_answer"]), axis=1)

# ── Calibrate semantic similarity to [0, 1] ───────────────────────────────────
# Stretch the empirical [min, max] to [0, 1] so the metric uses its full range.
sim_min = results_df["_raw_sim"].min()
sim_max = results_df["_raw_sim"].max()
rng = sim_max - sim_min if sim_max > sim_min else 1.0
results_df["semantic_score"] = ((results_df["_raw_sim"] - sim_min) / rng).clip(0, 1)
print(f"  Raw sim range: [{sim_min:.3f}, {sim_max:.3f}]  "
      f"→ calibrated to [0.000, 1.000]")

print("Computing AI judge scores (with chain-of-thought reasoning) ...")
results_df["ai_judge_score"] = results_df.apply(
    lambda r: ai_judge(r["question"], r["generated_answer"], r["expected_answer"]),
    axis=1)

# ── Final score ───────────────────────────────────────────────────────────────
results_df["final_score"] = (
    results_df["keyword_score"]  * 0.2 +
    results_df["semantic_score"] * 0.4 +
    results_df["ai_judge_score"] * 0.4
)

print("\n─── Results ───────────────────────────────────────────────────────")
print(f"  Keyword F1 (TF-IDF, mean)  : {results_df['keyword_score'].mean():.3f}")
print(f"  Semantic sim (calib, mean) : {results_df['semantic_score'].mean():.3f}  "
      f"(raw: {results_df['_raw_sim'].mean():.3f})")
print(f"  AI judge (CoT, mean)       : {results_df['ai_judge_score'].mean():.3f}")
print(f"  Final score (mean)         : {results_df['final_score'].mean():.3f}")
print(f"  Perplexity (mean)          : {results_df['perplexity'].mean():.1f}")
print("───────────────────────────────────────────────────────────────────")

best  = results_df.nlargest(3,  "final_score")[["question","expected_answer","generated_answer","final_score"]]
worst = results_df.nsmallest(3, "final_score")[["question","expected_answer","generated_answer","final_score"]]

print("\n── Top 3 answers ──")
for _, r in best.iterrows():
    print(f"  [{r['final_score']:.2f}]  Q: {r['question'][:70]}")
    print(f"         Expected : {r['expected_answer'][:70]}")
    print(f"         Generated: {r['generated_answer'][:70]}")
    print()

print("── Bottom 3 answers ──")
for _, r in worst.iterrows():
    print(f"  [{r['final_score']:.2f}]  Q: {r['question'][:70]}")
    print(f"         Expected : {r['expected_answer'][:70]}")
    print(f"         Generated: {r['generated_answer'][:70]}")
    print()

results_df.drop(columns=["_raw_sim"]).to_csv("results.csv", index=False)
print("Saved → results.csv")


Computing keyword F1 (TF-IDF weighted) ...
Computing raw semantic similarity ...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Raw sim range: [0.048, 1.000]  → calibrated to [0.000, 1.000]
Computing AI judge scores (with chain-of-thought reasoning) ...

─── Results ───────────────────────────────────────────────────────
  Keyword F1 (TF-IDF, mean)  : 0.221
  Semantic sim (calib, mean) : 0.506  (raw: 0.529)
  AI judge (CoT, mean)       : 0.590
  Final score (mean)         : 0.482
  Perplexity (mean)          : 15.4
───────────────────────────────────────────────────────────────────

── Top 3 answers ──
  [0.96]  Q: What are some examples of arrhythmias treated by antiarrhythmic agents
         Expected : Atrial fibrillation, supraventricular tachycardia, and ventricular tac
         Generated: Atrial fibrillation, supraventricular tachycardia, and ventricular tac

  [0.96]  Q: What are the non-specific signs that can support the diagnosis of FH?
         Expected : Xanthelasma and corneal arcus.
         Generated: Xanthelasma and corneal arcus.

  [0.91]  Q: What role does excessive alcohol consumption play 